# Настройка статических ключевых слов: задача (Русский)

Ноутбук демонстрирует полный цикл настройки словаря для тематического аспекта **задача/task** на RusErrC и сравнивает извлечение до настройки (`baseline`, без ключевых слов) и после настройки (`static-keywords`) на отложенной test-части.

Подробное описание алгоритма: `docs/keyword-tuning-algorithm.md`.

> Настройка использует эталоны только при построении train-evidence и выборе на dev. Test не участвует в выборе слов или стратегии.

## Режим запуска

- `sample`: быстрый воспроизводимый прогон на подвыборке.
- `full`: все записи RusErrC с непустым аспектом задачи и production-параметры.
- `TUNING_STRATEGIES`: список конкретных троек `(score, cluster, answer)`; `None` запускает полный перебор 27 комбинаций. Можно перечислить несколько стратегий, и лучшая будет выбрана по dev objective.

Evidence, метрики и checkpoints кэшируются. Повторный запуск с теми же параметрами продолжает поиск. Для русского языка требуются `pymorphy3`, NLTK stemmer и доступный sentence encoder. BERTScore включён в итоговое сравнение; его участие в objective задаётся отдельно.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import logging
import sys
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm


def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "untie").is_dir():
            return candidate
    raise RuntimeError("Не найден корень UnTIE_project")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if importlib.util.find_spec("pymorphy3") is None:
    raise RuntimeError("Русская настройка требует pymorphy3: установите requirements/dev.txt")

spec = importlib.util.spec_from_file_location(
    "untie_tuning_cli_ru", PROJECT_ROOT / "scripts" / "05_Tune_model_keywords.py"
)
if spec is None or spec.loader is None:
    raise RuntimeError("Не удалось загрузить scripts/05_Tune_model_keywords.py")
tuning_cli = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tuning_cli)

from untie.cli import load_static_keywords
from untie.extraction_metrics import METRIC_COLUMNS, compute_metrics_rowwise
from untie.keyword_evidence import CachedDocumentEvidence, rerank_cached_document
from untie.keyword_training import candidate_evidence_from_documents
from untie.keyword_tuning import aggregate_candidate_pool, deterministic_document_split

logging.basicConfig(level=logging.WARNING)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"Python = {sys.version.split()[0]}")

In [ ]:
# Главный переключатель: "sample" или "full".
RUN_MODE = "sample"
assert RUN_MODE in {"sample", "full"}

LANGUAGE = "ru"
SEED = 42
SAMPLE_SIZE = 20
DEVICE = "auto"
EVAL_BERTSCORE = True
TUNING_BERTSCORE = RUN_MODE == "full"

# None означает полный перебор 27 комбинаций. Чтобы проверить только выбранные
# стратегии, перечислите одну или несколько троек (score, cluster, answer).
TUNING_STRATEGIES = (
    [
        ("equal_weight_score_diff", "weighted_score", "combined_score"),
        # ("only_score_diff", "highest_avg_score", "highest_similarity"),
    ]
    if RUN_MODE == "sample"
    else None
)

VALID_SCORE_STRATEGIES = {"only_score_diff", "only_weight", "equal_weight_score_diff"}
VALID_CLUSTER_STRATEGIES = {"highest_avg_score", "weighted_score", "highest_cohesion"}
VALID_ANSWER_STRATEGIES = {"highest_chunk_score", "highest_similarity", "combined_score"}
for score_name, cluster_name, answer_name in TUNING_STRATEGIES or []:
    if score_name not in VALID_SCORE_STRATEGIES:
        raise ValueError(f"Неизвестная score strategy: {score_name}")
    if cluster_name not in VALID_CLUSTER_STRATEGIES:
        raise ValueError(f"Неизвестная cluster strategy: {cluster_name}")
    if answer_name not in VALID_ANSWER_STRATEGIES:
        raise ValueError(f"Неизвестная answer strategy: {answer_name}")

required_metric_modules = ["evaluate", "rouge_score"]
if EVAL_BERTSCORE or TUNING_BERTSCORE:
    required_metric_modules.append("bert_score")
missing_metric_modules = [
    name for name in required_metric_modules if importlib.util.find_spec(name) is None
]
if missing_metric_modules:
    raise RuntimeError(
        "Не установлены зависимости метрик: "
        f"{missing_metric_modules}. Установите проект с extras: pip install -e '.[evaluation]'"
    )

DATASET_PATH = PROJECT_ROOT / "datasets" / "ruserrc_structured.csv"
INIT_MODEL_PATH = PROJECT_ROOT / "model_params" / "ruserrc_init_model.json"
OUTPUT_DIR = (
    PROJECT_ROOT
    / "experiments"
    / "analysis_results"
    / "keyword_tuning_task"
    / LANGUAGE
    / RUN_MODE
)
CACHE_DIR = PROJECT_ROOT / "artifacts" / "keyword_tuning_notebooks"
TUNED_MODEL_PATH = OUTPUT_DIR / "ruserrc_tuned_model.json"
TRACE_PATH = OUTPUT_DIR / "ruserrc_tuned_model.tuning.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

profile_summary = {
    "mode": RUN_MODE,
    "dataset": str(DATASET_PATH),
    "sample_size": SAMPLE_SIZE if RUN_MODE == "sample" else "all",
    "strategies": TUNING_STRATEGIES or "all 27 combinations",
    "chunking": "128 tokens / 24 overlap",
    "tuning_bertscore": TUNING_BERTSCORE,
    "evaluation_bertscore": EVAL_BERTSCORE,
    "output": str(OUTPUT_DIR),
}
display(pd.Series(profile_summary, name="value").to_frame())

In [ ]:
dataset = tuning_cli.load_training_dataset(DATASET_PATH, LANGUAGE)
if RUN_MODE == "sample":
    dataset = dataset.iloc[:SAMPLE_SIZE].copy()
if len(dataset) < 5:
    raise ValueError("Для непустых train/dev/test требуется как минимум 5 документов")

split = deterministic_document_split(dataset["doc_id"], seed=SEED)
split_rows = []
for split_name, doc_ids in split.items():
    split_rows.append({"split": split_name, "documents": len(doc_ids), "example_ids": ", ".join(doc_ids[:3])})

display(pd.DataFrame(split_rows))
display(dataset[["doc_id", "tasks_cleaned"]].head())
print(f"Документов с непустым аспектом задачи: {len(dataset)}")
print("Пересечения split:", set(split["train"]) & set(split["dev"]), set(split["train"]) & set(split["test"]))

## Настройка

Следующая ячейка запускает официальный orchestration. Progress callback показывает сбор evidence, стратегии и stability runs. Для RU orchestration включает attention у QA-профиля явно и применяет `pymorphy3` для лемматизации.

In [ ]:
class NotebookProgress:
    def __init__(self) -> None:
        self.events: list[dict] = []
        self.evidence_bar = None
        self.strategy_bar = None
        self.stability_bar = None

    @staticmethod
    def _advance(bar, current: int) -> None:
        bar.update(max(0, current - bar.n))

    def __call__(self, event: str, current: int, total: int, details) -> None:
        record = {"event": event, "current": current, "total": total, **dict(details)}
        self.events.append(record)
        if event == "evidence":
            if self.evidence_bar is None:
                self.evidence_bar = tqdm(total=total, desc="Evidence (QA + attention)", unit="doc")
            self._advance(self.evidence_bar, current)
            self.evidence_bar.set_postfix(
                source=details.get("source"), candidates=details.get("candidate_count")
            )
            if current == total:
                self.evidence_bar.close()
        elif event == "candidate_pool":
            print(f"Пул кандидатов после train-only фильтрации: {details['candidate_count']}")
        elif event == "strategy_start":
            if self.strategy_bar is None:
                self.strategy_bar = tqdm(total=total, desc="Стратегии", unit="strategy")
            if self.stability_bar is not None:
                self.stability_bar.close()
            self.stability_bar = tqdm(
                total=None,
                desc=f"Stability: {details['strategy']}",
                unit="run",
                leave=False,
            )
        elif event == "stability_run":
            if self.stability_bar is not None:
                self.stability_bar.total = total
                self._advance(self.stability_bar, current)
                self.stability_bar.set_postfix(
                    objective=f"{details['objective']:.4f}", stop=details["stop_reason"]
                )
        elif event == "strategy_complete":
            if self.stability_bar is not None:
                self.stability_bar.close()
                self.stability_bar = None
            self._advance(self.strategy_bar, current)
            self.strategy_bar.set_postfix(objective=f"{details['objective']:.4f}")
            if current == total:
                self.strategy_bar.close()
        elif event == "complete":
            print(
                f"Готово: strategy={details['strategy']}, "
                f"keywords={len(details['keywords'])}, objective={details['objective']:.4f}"
            )


cli_args = [
    "--language", LANGUAGE,
    "--dataset", str(DATASET_PATH),
    "--model-params", str(INIT_MODEL_PATH),
    "--output", str(TUNED_MODEL_PATH),
    "--trace", str(TRACE_PATH),
    "--cache-dir", str(CACHE_DIR),
    "--device", DEVICE,
    "--seed", str(SEED),
]
if RUN_MODE == "sample":
    cli_args += [
        "--limit", str(SAMPLE_SIZE),
        "--min-document-support", "1",
        "--max-candidates", "30",
        "--max-keywords", "5",
        "--evaluation-budget", "30",
        "--patience", "1",
        "--beam-width", "3",
        "--stability-runs", "2",
        "--stability-threshold", "0.5",
    ]
for strategy_values in TUNING_STRATEGIES or []:
    cli_args.extend(["--strategy", *strategy_values])
if TUNING_BERTSCORE:
    cli_args.append("--include-bertscore")

args = tuning_cli.build_parser().parse_args(cli_args)
progress = NotebookProgress()
outcome = tuning_cli.run(args, progress_callback=progress)
progress_df = pd.DataFrame(progress.events)
display(progress_df.tail(10))

In [ ]:
summary = {
    "selected_strategy": outcome.strategy.name,
    "keywords": list(outcome.keywords),
    "dev_objective": outcome.objective,
    "dev_mean_gain": outcome.mean_gain,
    "dev_harm_rate": outcome.harm_rate,
    "dev_fallback_rate": outcome.fallback_rate,
    "stability": outcome.stability,
    "test_objective": outcome.test_objective,
    "test_mean_gain": outcome.test_mean_gain,
    "test_harm_rate": outcome.test_harm_rate,
    "test_fallback_rate": outcome.test_fallback_rate,
    "release_recommended": outcome.tuning_metadata()["release_recommended"],
}
display(pd.Series(summary, name="value").to_frame())

keyword_metadata_df = pd.DataFrame([item.to_dict() for item in outcome.keyword_metadata])
display(keyword_metadata_df)

evidence_dir = CACHE_DIR / LANGUAGE / "field-1" / "evidence"
evidence_documents = []
for path in sorted(evidence_dir.glob("*.json")):
    payload = json.loads(path.read_text(encoding="utf-8"))
    document = CachedDocumentEvidence.from_dict(payload)
    if document.doc_id in set(dataset["doc_id"]):
        evidence_documents.append(document)
evidence_by_id = {document.doc_id: document for document in evidence_documents}
missing = set(dataset["doc_id"]) - set(evidence_by_id)
if missing:
    raise RuntimeError(f"Не найден evidence для документов: {sorted(missing)[:5]}")

candidate_pool = aggregate_candidate_pool(
    candidate_evidence_from_documents(evidence_by_id[doc_id] for doc_id in split["train"]),
    split["train"],
    min_document_support=args.min_document_support,
)[: args.max_candidates]
candidate_df = pd.DataFrame([asdict(item) for item in candidate_pool])
display(candidate_df.head(20))

if not candidate_df.empty:
    top = candidate_df.head(15).sort_values("document_support")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].barh(top["term"], top["document_support"])
    axes[0].set_title("Train document support")
    axes[0].set_xlabel("documents")
    axes[1].scatter(candidate_df["attention"], candidate_df["score_diff"], alpha=0.65)
    selected = candidate_df[candidate_df["term"].isin(outcome.keywords)]
    axes[1].scatter(selected["attention"], selected["score_diff"], marker="*", s=180, label="selected")
    axes[1].set_title("Attention и semantic contrast")
    axes[1].set_xlabel("median attention")
    axes[1].set_ylabel("median score_diff")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
strategy_df = pd.DataFrame(
    [
        {
            "strategy": item.strategy.name,
            "objective": item.objective,
            "mean_gain": item.mean_gain,
            "harm_rate": item.harm_rate,
            "fallback_rate": item.fallback_rate,
            "stability": item.stability,
            "keyword_count": len(item.keywords),
            "evaluations_used": item.evaluations_used,
            "stop_reasons": " | ".join(item.stop_reasons),
        }
        for item in outcome.strategy_outcomes
    ]
)
display(strategy_df.head(27))
display(pd.DataFrame([asdict(item) for item in outcome.ablations]))

best_strategy = next(item for item in outcome.strategy_outcomes if item.strategy == outcome.strategy)
trace_rows = []
for run_index, trace in enumerate(best_strategy.traces, start=1):
    for step in trace:
        trace_rows.append({"run": run_index, **asdict(step)})
trace_df = pd.DataFrame(trace_rows)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ranked = strategy_df.sort_values("objective").tail(12)
axes[0].barh(ranked["strategy"], ranked["objective"])
axes[0].set_title("Лучшие стратегии по dev objective")
axes[0].set_xlabel("objective")
if not trace_df.empty:
    for run_index, group in trace_df.groupby("run"):
        axes[1].plot(group["evaluations_used"], group["objective"], marker="o", label=f"run {run_index}")
    axes[1].legend()
axes[1].set_title("Ход SFFS для выбранной стратегии")
axes[1].set_xlabel("evaluations used")
axes[1].set_ylabel("objective")
plt.tight_layout()
plt.show()

## Baseline против настроенной модели на held-out test

Сравнение использует сохранённые QA-ответы test-документов. `baseline_answer` получен без фильтрации и ключей. `rerank_cached_document` эквивалентен live `StaticKeywordRerankingPipeline` при тех же ключах и стратегии, поэтому повторный QA не требуется.

In [ ]:
weighted_keywords, selected_strategy = load_static_keywords(TUNED_MODEL_PATH, field_id=1)
weight_ratio = {
    "only_score_diff": 0.0,
    "only_weight": 1.0,
    "equal_weight_score_diff": 0.5,
}.get(str(selected_strategy.get("score_chunk_strategy", "")), 0.5)
cluster_strategy = str(selected_strategy.get("choose_cluster_strategy", "weighted_score"))
answer_strategy = str(selected_strategy.get("choose_answer_strategy", "combined_score"))

comparison_rows = []
for doc_id in tqdm(outcome.test_doc_ids, desc="Baseline vs tuned", unit="doc"):
    document = evidence_by_id[doc_id]
    tuned_answer, diagnostics = rerank_cached_document(
        document,
        weighted_keywords,
        weight_ratio=weight_ratio,
        cluster_strategy=cluster_strategy,
        answer_strategy=answer_strategy,
    )
    common = {"doc_id": doc_id, "tasks_cleaned": list(document.references)}
    comparison_rows.append(
        {**common, "mode": "baseline", "prediction": document.baseline_answer, "fallback": False}
    )
    comparison_rows.append(
        {
            **common,
            "mode": "static-keywords",
            "prediction": tuned_answer,
            "fallback": bool(diagnostics.get("fallback", False)),
        }
    )

comparison = pd.DataFrame(comparison_rows)
display(comparison.head(8))
print(f"Test documents: {len(outcome.test_doc_ids)}")
tuned_fallback_rate = comparison.loc[
    comparison["mode"] == "static-keywords", "fallback"
].mean()
print(f"Keyword fallback rate: {tuned_fallback_rate:.3f}")

In [ ]:
scored = compute_metrics_rowwise(
    comparison,
    pred_col="prediction",
    gold_col="tasks_cleaned",
    lang=LANGUAGE,
    include_bertscore=EVAL_BERTSCORE,
    show_progress=True,
)
metric_columns = [column for column in METRIC_COLUMNS if column in scored.columns]
quality_parts = [scored["char_f1"], scored["token_f1"] / 100.0, scored["rouge_l_f1"]]
if EVAL_BERTSCORE:
    quality_parts.append(scored["bertscore_f1"])
scored["composite_quality"] = sum(quality_parts) / len(quality_parts)

aggregate = scored.groupby("mode")[metric_columns + ["composite_quality"]].mean()
display(aggregate)

wide_quality = scored.pivot(index="doc_id", columns="mode", values="composite_quality")
wide_quality["delta"] = wide_quality["static-keywords"] - wide_quality["baseline"]
epsilon = 1e-9
win_tie_loss = pd.Series(
    {
        "win": int((wide_quality["delta"] > epsilon).sum()),
        "tie": int((wide_quality["delta"].abs() <= epsilon).sum()),
        "loss": int((wide_quality["delta"] < -epsilon).sum()),
        "harm_rate": float((wide_quality["delta"] < -0.01).mean()),
        "mean_delta": float(wide_quality["delta"].mean()),
    },
    name="value",
)
display(win_tie_loss.to_frame())

scored.to_csv(OUTPUT_DIR / "baseline_vs_static_keywords_per_document.csv", index=False)
aggregate.to_csv(OUTPUT_DIR / "baseline_vs_static_keywords_summary.csv")
wide_quality.to_csv(OUTPUT_DIR / "composite_quality_deltas.csv")
print(f"Результаты сохранены в {OUTPUT_DIR}")

In [ ]:
plot_data = aggregate[metric_columns].copy()
if "token_f1" in plot_data.columns:
    plot_data["token_f1"] = plot_data["token_f1"] / 100.0

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_data.T.plot(kind="bar", ax=axes[0])
axes[0].set_title("Среднее качество на held-out test")
axes[0].set_ylabel("score (token_f1 нормирован в [0, 1])")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=35)
axes[0].legend(title="mode")

axes[1].axhline(0, color="black", linewidth=1)
axes[1].bar(np.arange(len(wide_quality)), wide_quality["delta"].sort_values().values)
axes[1].set_title("Изменение composite quality по документам")
axes[1].set_xlabel("test documents (sorted)")
axes[1].set_ylabel("static-keywords − baseline")
plt.tight_layout()
plt.show()

prediction_wide = scored.pivot(index="doc_id", columns="mode", values="prediction")
examples = prediction_wide.join(wide_quality[["delta"]]).sort_values("delta", ascending=False)
print("Наибольшие улучшения:")
display(examples.head(5))
print("Наибольшие ухудшения:")
display(examples.tail(5).sort_values("delta"))

## Интерпретация

Перед использованием tuned-модели проверяйте положительный test gain, bootstrap lower bound, harm/fallback rates, stability и конкретные примеры ухудшений. `release_recommended` — автоматический gate, а не замена ручной проверке.

Для production-вывода установите `RUN_MODE = "full"`. Начальная модель RusErrC не изменяется: tuned JSON, trace и таблицы сохраняются отдельно.